# [stand-in] talk adapter — v8t only: SFT with Unsloth, export to GGUF

**What this is.** The second of v8's two adapters. The trunk (`v8e`, the program adapter, its own notebook
`standin_emitter_sft.ipynb`) emits CubeLang programs; **this notebook trains the talk cortex** — CubbyChat,
identity, content awareness, emotion, affect, history, safety — as a separate adapter behind the same
`Emitter.emit(…, context="talk")` interface. Nothing it measures is a CubbyLLM result: tag every number
`[stand-in]`; it lives in `standin/`, outside `cubbyllm/`.

**Data (v9t, 2026-09-04).** `Drive/cubbyllm/standin/emitter_sft_v9t.jsonl` (+ `.manifest.json`, `partition: talk`): **89,890 records** — chat 28,144 (8192-context caps, OASST2 second exchanges inline, Claire, DiaBLa, ReDial, the owner's WhatsApp/Teams) · history 12,916 · emotion 9,879 · quebec 4,795 + quebec_mc 4,804 · repair 4,986 · rewrite 4,997 · appraisal 4,990 · safety 2,772 · dialog_emotion 2,847 · dialog_act 1,978 · empathy 2,500 · affect 1,679 · content 1,396 · verbalize 621 · identity 586; every record carries its own `system` prompt and a `repeat` weight. No programs: the VM is not needed here. The v8t set (47,431 records) remains the control's data.

**Data (v8t).** `Drive/cubbyllm/standin/emitter_sft_v8t.jsonl` (+ `.manifest.json`, `partition: talk`, built by
`standin/data/build_chat_sft.py --version v8 --partition both`): **47,431 records** — chat 20,569 · history 12,941 ·
emotion 9,880 · affect 1,679 · content 1,396 · identity 586 · safety 380; train 45,051 / val 2,380; every record
carries its own `system` prompt (identity turns carry the hormonal state) and a `repeat` weight (identity ×4,
the replayed game families ×2). No programs: the VM is not needed here.

**The base model is the one knob.** The `LiquidAI/LFM2.5-2.6B` run is the **control** and runs first — it is
the v8 measurement (does the split hold chat/identity/emotion/affect/history at v7 or better?). The bake-off
candidates (`standin/README.md` § v8, "Which base for talk") run from the same notebook by changing `MODEL`
in the setup cell; each base gets its own artifact dir. The think-block prefill and the batch shape follow the
base automatically; the serve-side emitter picks the chat-template family from the GGUF the same way
(`standin/emitter.py::chat_family`).

**What to bring back.** `{OUT}/gguf/*q4_k_m.gguf` (serve it with `--talk-gguf`) and `{OUT}/val_generations.json`
(the 40/task talk read; replay it locally with `standin/eval_emitter_vm.py --val-generations …`). The decision
rule for a non-LFM base: `identity_ok` at 1.0 EN+FR **and** chat/history/emotion up by ≥ 5 points at n=40
**and** it fits the 12 GB card beside the 2.6B emitter — otherwise LFM v8t stays.

**Eval only.** After a VM reset the merged model is still on Drive: set `os.environ['STANDIN_EVAL_ONLY']='1'`
before the setup cell; cell 2 loads `{OUT}/merged` and cells 3–4 are skipped.


In [ ]:
# --- setup (run once per session) ---
import os, json, time, random, re
!pip -q install unsloth trl datasets
import sys
from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cubbyllm/standin'
# the repo checkout (loud: a failed clone/pull used to hide behind -q and surface as 'cannot find identity')
if not os.path.exists('/content/CubbyLLM/.git'):
    !rm -rf /content/CubbyLLM && git clone https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM 2>&1 | tail -2
else:
    !cd /content/CubbyLLM && git fetch origin 2>&1 | tail -1 && git reset -q --hard origin/master && git log -1 --format='repo at %h %s'
sys.path.insert(0, '/content/CubbyLLM/standin/data'); sys.path.insert(0, '/content/CubbyLLM')
try:
    from identity import identity_ok, load_facts, EMITTER_SYSTEM, voice_ok, is_model_guard, is_identity_reply   # the identity check + facts
    print('identity: from the repo checkout')
except ImportError as e:                        # no checkout (private repo / network) OR a stale copy: Drive carries a copy
    assert os.path.exists(f'{DRIVE}/identity.py'), f'identity.py not in the repo checkout nor at {DRIVE} ({e})'
    sys.modules.pop('identity', None); sys.path.insert(0, DRIVE)
    from identity import identity_ok, load_facts, EMITTER_SYSTEM, voice_ok, is_model_guard, is_identity_reply
    print('identity: from Drive (repo checkout missing or stale:', e, ')')
import identity as _idn
assert all(callable(getattr(_idn, n, None)) for n in ('is_model_guard', 'is_identity_reply', 'voice_ok', 'identity_ok')), f'{_idn.__file__} is stale -- copy standin/data/identity.py from the repo to {DRIVE}. NEVER substitute placeholders: the chat/identity scores would be meaningless.'
print('identity module:', _idn.__file__)
FACTS = load_facts()

VERSION = os.environ.get('STANDIN_VERSION', 'v9t')   # this notebook trains the TALK adapter only; v9t = v8t + the gap families
DATA = f'{DRIVE}/emitter_sft_{VERSION}.jsonl'
MANIFEST = f'{DRIVE}/emitter_sft_{VERSION}.manifest.json'

# --- the one knob: which base. The LFM run is the CONTROL and runs first; the others are the bake-off ---
MODEL = os.environ.get('STANDIN_MODEL', 'LiquidAI/LFM2.5-2.6B')
# MODEL = 'LiquidAI/LFM2.5-8B-A1B'          # same family, MoE 1.5B active of 8.3B: the speed candidate (ChatML + think block, like the 2.6B)
# MODEL = 'google/gemma-4-E4B-it'           # the chat-quality candidate (Gemma template, no think block; a multimodal base used text-only)
# MODEL = 'Qwen/Qwen3-4B-Instruct-2507'     # the 4B text-only candidate (ChatML, no think block)
MODEL_TAG = '' if MODEL == 'LiquidAI/LFM2.5-2.6B' else '_' + MODEL.split('/')[-1].lower().replace('.', 'p')
OUT = f'{DRIVE}/emitter_lfm25_2p6b_{VERSION}' + MODEL_TAG   # adapter + merged + GGUF; a non-default base never overwrites the LFM run
THINKS = MODEL.startswith('LiquidAI/LFM2.5-')      # LFM2.5 opens a <think> block on its own: targets and prefill close it immediately
NO_THINK = '<think>\n</think>\n' if THINKS else ''   # the serve emitter applies the same rule from the GGUF's architecture (family "lfm")
IS_CONTROL = MODEL == 'LiquidAI/LFM2.5-2.6B'
BATCH, ACCUM = (32, 1) if IS_CONTROL else (8, 4)   # the 2.6B dense fits 32x4096 in one micro-batch on the A100-80G (v6: ~8 GB); a bigger or MoE base keeps the effective 32 by accumulation
EPOCHS = 2                                          # the talk set plateaus at its mixture floor after 1 epoch (v6); 2 epochs is what v7 got
LORA_R = 64                                         # owner (2026-09-03): r 64 / alpha 2r for the multilingual talk set; the same recipe on every arm
# LoRA targets per base. LFM2/LFM2.5: attention q/k/v/out_proj, short-conv in_proj/out_proj, dense FFN w1/w2/w3; the MoE experts
# (gate_up_proj/down_proj, bare Parameters) are auto-added by Unsloth via target_parameters. NOT `gate`: that is the top-k router
# (a custom module PEFT cannot wrap -- it raises -- and training the router is not wanted). Gemma/Qwen use the standard names.
TARGETS = (['q_proj', 'k_proj', 'v_proj', 'out_proj', 'in_proj', 'w1', 'w2', 'w3'] if MODEL.startswith('LiquidAI/LFM2')
           else ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
EVAL_ONLY = os.environ.get('STANDIN_EVAL_ONLY') == '1'   # 1 = load {OUT}/merged from Drive, skip LoRA/train/export, run the eval only
if EVAL_ONLY: print('EVAL_ONLY: will load', f'{OUT}/merged', '(exists:', os.path.exists(f'{OUT}/merged'), ')')
MAX_SEQ = 8192   # owner (2026-09-03): the talk set keeps its long rows whole (v8t recipe); rows pad to the longest in the batch, not to MAX_SEQ
print('base', MODEL, '| control' if IS_CONTROL else '| bake-off candidate', '| think block:', THINKS, '| batch', BATCH, 'x', ACCUM, '| epochs', EPOCHS, '| out', OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv
for f in (DATA, MANIFEST):
    print(('ok      ' if os.path.exists(f) else 'MISSING ') + f)
m = json.load(open(MANIFEST))
assert m.get('partition') == 'talk', f"{MANIFEST} is not the talk partition (partition={m.get('partition')!r}); build it with build_chat_sft.py --version v8 --partition both"
print('manifest', m.get('version'), m.get('partition'), ':', m['by_task'], '| records', m.get('n_records'),
      '| built', m['built'][:19], '| git', m['git_rev'][:8])


In [ ]:
# --- data: chat-format the records; train/val from the builder's deterministic split ---
SYSTEM = EMITTER_SYSTEM   # fallback only: every talk record carries its own r['system'] (identity turns include the hormonal state)
recs = [json.loads(l) for l in open(DATA, encoding='utf-8')]
recs = [r for r in recs if r.get('vm_ok') in (True, None) and r.get('gold_match') is not False]
from gap_families import GAP_TASKS, gap_ok   # v9: the gap families and their checks (repo checkout)
TALK_TASKS = ('identity', 'chat', 'content', 'emotion', 'affect', 'history', 'safety', 'exposure') + GAP_TASKS
assert all(r['task'] in TALK_TASKS for r in recs), 'a program record in the talk partition'
train = [r for r in recs if r['split'] == 'train']; val = [r for r in recs if r['split'] == 'val']
from collections import Counter
print('train', len(train), Counter(r['task'] for r in train)); print('val  ', len(val), Counter(r['task'] for r in val))

def to_messages(r):
    return [{'role': 'system', 'content': r.get('system') or SYSTEM},
            {'role': 'user', 'content': r['prompt']},
            {'role': 'assistant', 'content': NO_THINK + r['program'].strip() + '\n'}]

try:
    from unsloth import FastModel as Loader          # text and multimodal bases (Gemma 4) through one loader
except ImportError:
    from unsloth import FastLanguageModel as Loader
model, tokenizer = Loader.from_pretrained((f'{OUT}/merged' if EVAL_ONLY else MODEL), max_seq_length=MAX_SEQ, load_in_4bit=False, dtype=None)
print('model:', f'{OUT}/merged (trained, from Drive)' if EVAL_ONLY else MODEL)
tpl = tokenizer.chat_template or ''
family = 'gemma' if '<start_of_turn>' in tpl else ('lfm' if THINKS else 'chatml')
print('chat template family:', family, '| template head:', tpl[:300].replace('\n', ' '))   # the serve emitter must agree: standin/emitter.py::chat_family

def fmt(r):
    return {'text': tokenizer.apply_chat_template(to_messages(r), tokenize=False, add_generation_prompt=False)}
from datasets import Dataset
random.Random(0).shuffle(train)
ds_train = Dataset.from_list([fmt(r) for r in train for _ in range(int(r.get('repeat', 1)))])   # identity x4, replayed game families x2 (builder's `repeat`)
print('train rows after repeat weights:', len(ds_train))
lens = [len(tokenizer(x['text']).input_ids) for x in ds_train.select(range(min(500, len(ds_train))))]
print('token lengths (sample of 500): max', max(lens), 'p95', sorted(lens)[int(0.95*len(lens))], '-> MAX_SEQ', MAX_SEQ)
print(ds_train[0]['text'][:900])


In [ ]:
# --- LoRA + SFT ---
if EVAL_ONLY:
    print('EVAL_ONLY: skipping LoRA + training; the merged model from Drive is already loaded')
else:
    from trl import SFTTrainer, SFTConfig
    # v8t recipe (owner, 2026-09-03): alpha 64 (scale 2), LR 1e-4, 10% warmup, 8-bit AdamW -- the emitter's 2e-4 / alpha 32 /
    # 20-step-warmup recipe was unstable on the talk set. `optim` belongs to SFTConfig: get_peft_model forwards unknown kwargs
    # into LoraConfig, which raises on it. MoE bases (LFM2.5-8B-A1B): Unsloth auto-populates target_parameters for the expert
    # weights, so the same call covers the experts.
    model = Loader.get_peft_model(
        model, r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.0, bias='none',
        target_modules=TARGETS, use_gradient_checkpointing='unsloth', random_state=3407)
    # effective batch 32 for every base (BATCH x ACCUM from the setup cell); v6's 0.5 floor was the mixture floor of free
    # chat/history text, not a batch-size effect -- epochs and the talk checks decide, not the loss.
    # warmup: transformers 5 dropped warmup_ratio and reads a float warmup_steps as a ratio; older versions want warmup_ratio.
    WARMUP = {'warmup_ratio': 0.05} if 'warmup_ratio' in SFTConfig.__dataclass_fields__ else {'warmup_steps': 0.05}
    # packing stays OFF: TRL's packing goes padding-free (FA2/3 only) and LFM2's short-conv layers see no record boundaries,
    # so packed records bleed into each other through the conv path whatever attention does; it also changes the comparison with v7.
    cfg = SFTConfig(output_dir='/content/talk_ckpt', per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
                    num_train_epochs=EPOCHS, learning_rate=1e-4, lr_scheduler_type='cosine', weight_decay=0.01, **WARMUP,
                    optim='adamw_8bit', logging_steps=10, save_strategy='no', bf16=True, max_seq_length=MAX_SEQ,
                    dataset_text_field='text', packing=False, report_to='none', seed=0)
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds_train, args=cfg)
    t0 = time.time(); stats = trainer.train(); print(f'trained in {(time.time()-t0)/60:.1f} min; final loss', stats.training_loss)
    os.makedirs(OUT, exist_ok=True); model.save_pretrained(f'{OUT}/adapter'); tokenizer.save_pretrained(f'{OUT}/adapter')
    print('adapter ->', f'{OUT}/adapter')


In [ ]:
# --- export FIRST: merged fp16 + GGUF (q4_k_m serves on the 12 GB Vulkan box; q8_0 too for the control, the fair comparison) ---
if EVAL_ONLY:
    print('EVAL_ONLY: skipping export; the merged model from Drive is already loaded')
else:
    QUANTS = ['q4_k_m'] + (['q8_0'] if IS_CONTROL else [])   # a bigger base's q8 is Drive space for nothing: q4 is what serves
    model.save_pretrained_merged(f'{OUT}/merged', tokenizer, save_method='merged_16bit')
    model.save_pretrained_gguf(f'{OUT}/gguf', tokenizer, quantization_method=QUANTS)
    import glob as _glob
    GGUFS = sorted(_glob.glob(f'{OUT}/gguf*/*.gguf'))   # recent Unsloth writes the GGUFs to {OUT}/gguf_gguf and an fp16 HF copy to {OUT}/gguf
    print('GGUF files:'); [print('  ', g, f'{os.path.getsize(g)/1e9:.2f} GB') for g in GGUFS]
    print('(the fp16 copies in', f'{OUT}/gguf', 'and', f'{OUT}/merged', 'are Drive space once the GGUFs exist)')
    print('done -> download the q4_k_m GGUF; serve: serve_api.py --gguf <v8e Q4> --talk-gguf <this Q4> --pacman')


In [ ]:
import re as _re
def affect_ok(r, g, tol=0.35):   # mirrors standin/data/build_chat_sft.py::affect_ok
    m = _re.search(r'valence\s*[:=]?\s*([+-]?\d*\.?\d+)', g or '', _re.I); n = _re.search(r'arousal\s*[:=]?\s*([+-]?\d*\.?\d+)', g or '', _re.I)
    nums = [float(m.group(1)), float(n.group(1))] if (m and n) else [float(x) for x in _re.findall(r'[+-]?\d*\.?\d+', g or '')[:2]]
    v, a = (r.get('gold_any') or [None, None])[:2]
    return len(nums) == 2 and v is not None and abs(nums[0] - v) <= tol and abs(nums[1] - a) <= tol
def history_ok(r, g):   # mirrors standin/data/build_chat_sft.py::history_ok
    if not g or not voice_ok(g, FACTS) or is_model_guard(g): return False
    sub, gold = r.get('subtype'), str(r.get('gold') or '')
    if sub == 'dating':
        m = _re.search(r'\b(\d{4})\b', g); return bool(m) and gold[:4].isdigit() and abs(int(m.group(1)) - int(gold[:4])) <= 5
    if sub == 'when': return _re.sub(r'\s*BCE?$', '', gold) in g
    return any(str(x).lower() in g.lower() for x in (r.get('gold_any') or []))
# --- the talk read: a stratified 40/task on the held-out val split (the bake-off's number; the local eval replays this file) ---
Loader.for_inference(model)
import transformers; transformers.logging.set_verbosity_error()
def strip_think(s):   # LFM2.5's template opens a <think> block; everything up to </think> is reasoning, not the answer
    return re.sub(r'^\s*(?:<think>)?.*?</think>\s*', '', s, count=1, flags=re.S) if '</think>' in s else s
STOP = ['<end_of_turn>'] if family == 'gemma' else ['<|im_end|>']
def emit(prompt, max_new=200, system=None):
    text = tokenizer.apply_chat_template([{'role': 'system', 'content': system or SYSTEM}, {'role': 'user', 'content': prompt}],
                                         tokenize=False, add_generation_prompt=True) + NO_THINK   # prefill: no reasoning (LFM only)
    enc = tokenizer(text, return_tensors='pt', add_special_tokens=False).to('cuda')
    out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None,
                         pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    for s in STOP:
        gen = gen.split(s)[0]
    return gen
import torch
def val_loss_by_task(model, tokenizer, val, to_messages, no_think, n_per_task=60, batch=8):
    # mean per-token NLL on the ASSISTANT span (prompt masked) per task -- the loss the trainer averages away
    import torch
    by = {}
    for r in val: by.setdefault(r['task'], []).append(r)
    rng_l = random.Random(2); out = {}
    side = tokenizer.padding_side; tokenizer.padding_side = 'right'
    if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
    try:
        for task, rs in sorted(by.items()):
            rs = rng_l.sample(rs, min(n_per_task, len(rs))); tot_nll, tot_tok = 0.0, 0
            for i in range(0, len(rs), batch):
                chunk = rs[i:i + batch]
                fulls = [tokenizer.apply_chat_template(to_messages(r), tokenize=False, add_generation_prompt=False) for r in chunk]
                plens = [len(tokenizer(tokenizer.apply_chat_template(to_messages(r)[:2], tokenize=False, add_generation_prompt=True) + no_think,
                                       add_special_tokens=False).input_ids) for r in chunk]
                enc = tokenizer(fulls, return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
                labels = enc['input_ids'].clone()
                for j, pl in enumerate(plens): labels[j, :pl] = -100
                labels[enc['attention_mask'] == 0] = -100
                with torch.no_grad():
                    logits = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask']).logits
                sl, sb = logits[:, :-1].float(), labels[:, 1:]
                tot_nll += torch.nn.functional.cross_entropy(sl.reshape(-1, sl.size(-1)), sb.reshape(-1), ignore_index=-100, reduction='sum').item()
                tot_tok += int((sb != -100).sum())
            out[task] = {'nll': round(tot_nll / max(1, tot_tok), 4), 'tokens': tot_tok, 'n': len(rs)}
    finally:
        tokenizer.padding_side = side
    return out
VAL_LOSS = val_loss_by_task(model, tokenizer, val, to_messages, NO_THINK)
print('[stand-in] val loss by task (nats/token on the assistant span; chat/history sit at the human-text floor ~0.5, label families near 0):')
for t_, v_ in sorted(VAL_LOSS.items(), key=lambda kv: -kv[1]['nll']): print(f"  {t_:15s} {v_['nll']:6.3f}  ({v_['tokens']} tokens, {v_['n']} records)")
N_PER_TASK = int(os.environ.get('STANDIN_EVAL_N', '40'))   # 40 = the stratified read (identity has 32 val turns, safety 20: all of them)
rng = random.Random(1); by_task = {}
for r in val: by_task.setdefault(r['task'], []).append(r)
sample = [r for t, rs in sorted(by_task.items()) for r in rng.sample(rs, min(N_PER_TASK, len(rs)))]
CAP = {'identity': 200, 'chat': 120, 'content': 40, 'emotion': 40, 'history': 160, 'affect': 40, 'safety': 40, 'exposure': 40,
       'verbalize': 48, 'repair': 60, 'rewrite': 60, 'appraisal': 24, 'dialog_emotion': 24, 'dialog_act': 12, 'empathy': 12,
       'quebec': 60, 'quebec_mc': 24}
print('eval sample', len(sample), {t: min(N_PER_TASK, len(rs)) for t, rs in sorted(by_task.items())})
hits = Counter(); tot = Counter(); by_lang = Counter(); by_lang_hit = Counter(); outputs = []
t0 = time.time()
for i, r in enumerate(sample):
    gen = emit(r['prompt'], max_new=CAP.get(r['task'], 120), system=r.get('system')); g = strip_think(gen).strip()
    if r['task'] == 'identity':  ok = identity_ok(r.get('subtype', ''), g, FACTS, r.get('lang', 'en'))
    elif r['task'] == 'chat':    ok = bool(g) and voice_ok(g, FACTS) and not is_model_guard(g) and not is_identity_reply(g, FACTS)
    elif r['task'] == 'emotion': ok = g.lower().replace('—', ',').split(',')[0].strip(' -:.') in {str(x).lower() for x in (r.get('gold_any') or [r.get('gold')])}
    elif r['task'] == 'history': ok = history_ok(r, g)
    elif r['task'] == 'affect':  ok = affect_ok(r, g)
    elif r['task'] in GAP_TASKS: ok = gap_ok(r, g, FACTS)
    else:                        ok = g.lower().split(' ')[0].strip(' —-:.,') == str(r.get('gold')).lower()   # content / safety / exposure: label first
    tot[r['task']] += 1; hits[r['task']] += int(ok)
    lang = r.get('lang') or 'en'; by_lang[lang] += 1; by_lang_hit[lang] += int(ok)
    outputs.append({'id': r['id'], 'task': r['task'], 'subtype': r.get('subtype', ''), 'prompt': r['prompt'],
                    'reference': r['program'], 'generated': gen, 'exact_match': ok, 'gold': r.get('gold'), 'gold_any': r.get('gold_any'),
                    'system': r.get('system'), 'lang': r.get('lang')})
    if (i+1) % 50 == 0: print(f'  {i+1}/{len(sample)} ({time.time()-t0:.0f}s)')
print('[stand-in] talk val by task (identity = identity_ok; chat = voice/guard/no-bio; emotion/content/safety = label first; history/affect/gap families = their checks):',
      {t: f'{hits[t]}/{tot[t]}' for t in tot}, '| by lang', {l: f'{by_lang_hit[l]}/{by_lang[l]}' for l in by_lang}, '| overall', round(sum(hits.values())/sum(tot.values()), 3))
json.dump({'model': MODEL, 'version': VERSION, 'family': family, 'no_think': bool(NO_THINK), 'n': len(sample), 'val_loss_by_task': VAL_LOSS,
           'exact_match_by_task': {t: hits[t]/tot[t] for t in tot}, 'by_lang': {l: by_lang_hit[l]/by_lang[l] for l in by_lang},
           'outputs': outputs, 'manifest_output_sha256': m['output_sha256']}, open(f'{OUT}/val_generations.json', 'w'), indent=1)
print('generations ->', f'{OUT}/val_generations.json  (replay locally: standin/eval_emitter_vm.py --data standin/data/out/emitter_sft_v8t.jsonl --val-generations <this file>)')


### How to read

- **Per task, EN and FR apart.** `identity` is scored by `identity_ok` (name/builder present, never AGI or feelings,
  never another model, the verbatim don't-know line in the question's language); `chat` by the voice rules
  (no guard phrasing, no biography when none was asked); `emotion`/`content`/`safety` by the label coming first;
  `history` and `affect` by the same checks the builder uses. None of these is exact match.
- **The control's bar is v7 at 40/task**: chat 1.0, content 1.0, identity 0.867, emotion 0.675, affect 0.975, history 0.625
  (VM-verified, `standin/README.md`). The v8t control read (2026-09-03): chat/content/affect/safety 1.0, identity 0.938,
  emotion 0.600, history 0.500 — behaviour at ceiling, the recall families (dating, news, quote_who) dipped. The split is worth its VRAM only if the *program* adapter (`v8e`) brings arithmetic back
  to ≥ 0.825 with forge 1.00 and self-test 96/4/0 while this adapter holds these.
- **A candidate base replaces LFM for talk only if** `identity_ok` stays 1.0 EN+FR, chat/history/emotion gain ≥ 5 points
  at n=40, and its Q4 fits beside the 2.6B emitter on the 12 GB card. A base that loses the identity voice is out
  whatever its chat score.
- **Think block.** Only the LFM bases open `<think>`; their targets and the eval prefill close it at once and scoring
  strips it. Other bases train and score on the plain answer. The serve emitter chooses the same way from the GGUF.
- **Everything here is `[stand-in]`.** It goes in `standin/README.md` and the TODO, never in `CUBBYLLM_HYPOTHESES.md`
  except as a pointer.
